# Reset Notebook Configurations

In [ ]:
# Restart the Runtime (Hard Reset)
# This is the most effective way to completely clear RAM and disk cache.
import os
os.kill(os.getpid(), 9)

In [ ]:
# Delete variables
%reset -f

# Clear CUDA cache (only for deep learning examples)
import torch
torch.cuda.empty_cache()

# Clear garbage
import gc
gc.collect()

30

In [ ]:
!rm -rf /content/*
!rm -rf ~/.cache/huggingface

In [ ]:
!df -h       # Disk usage
print("="*100)
print("="*100)
!nvidia-smi  # GPU usage
print("="*100)
print("="*100)
!free -h     # RAM usage

Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   65G  43% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  750M  62% /usr/sbin/docker-init
/dev/sda1       119G   72G   48G  61% /opt/bin/.nvidia
tmpfs           6.4G  7.6M  6.4G   1% /var/colab
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware
Fri Nov 14 04:26:33 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|              

# Code Run

In [1]:
# Install uv package manager
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 99.6 MB/s eta 0:00:00


In [2]:
!uv pip install gradio pandas numpy plotly

Using Python 3.12.12 environment at: /usr
Audited 4 packages in 123ms


In [5]:
import gradio as gr
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta

# Generate sample sales dataset
def generate_sales_data():
    np.random.seed(42)

    # Date range: last 12 months
    end_date = datetime.now()
    start_date = end_date - timedelta(days=365)
    dates = pd.date_range(start=start_date, end=end_date, freq='D')

    # Products and regions
    products = ['Laptop', 'Smartphone', 'Tablet', 'Headphones', 'Smartwatch']
    regions = ['North America', 'Europe', 'Asia', 'South America']

    # Generate random sales data
    data = []
    for date in dates:
        for product in products:
            for region in regions:
                # Add some seasonality and randomness
                base_sales = np.random.randint(10, 100)
                seasonal_factor = 1 + 0.3 * np.sin(2 * np.pi * date.dayofyear / 365)
                sales = int(base_sales * seasonal_factor)
                revenue = sales * np.random.uniform(200, 1500)

                data.append({
                    'Date': date,
                    'Product': product,
                    'Region': region,
                    'Units_Sold': sales,
                    'Revenue': round(revenue, 2)
                })

    return pd.DataFrame(data)

# Load the dataset
df = generate_sales_data()

def analyze_sales(product_filter, region_filter, start_date, end_date):
    # Filter data based on selections
    filtered_df = df.copy()

    if product_filter != "All":
        filtered_df = filtered_df[filtered_df['Product'] == product_filter]

    if region_filter != "All":
        filtered_df = filtered_df[filtered_df['Region'] == region_filter]

    if start_date and end_date:
        filtered_df = filtered_df[
            (filtered_df['Date'] >= pd.Timestamp(start_date)) &
            (filtered_df['Date'] <= pd.Timestamp(end_date))
        ]

    # Calculate metrics
    total_revenue = filtered_df['Revenue'].sum()
    total_units = filtered_df['Units_Sold'].sum()
    avg_revenue = filtered_df['Revenue'].mean()

    # Create visualizations
    # 1. Revenue over time
    daily_revenue = filtered_df.groupby('Date')['Revenue'].sum().reset_index()
    fig1 = px.line(daily_revenue, x='Date', y='Revenue',
                   title='Revenue Over Time',
                   labels={'Revenue': 'Revenue ($)'})
    fig1.update_traces(line_color='#1f77b4', line_width=2)

    # 2. Sales by Product
    product_sales = filtered_df.groupby('Product')['Revenue'].sum().reset_index()
    product_sales = product_sales.sort_values('Revenue', ascending=False)
    fig2 = px.bar(product_sales, x='Product', y='Revenue',
                  title='Revenue by Product',
                  labels={'Revenue': 'Revenue ($)'},
                  color='Revenue',
                  color_continuous_scale='Blues')

    # 3. Sales by Region (Pie Chart)
    region_sales = filtered_df.groupby('Region')['Revenue'].sum().reset_index()
    fig3 = px.pie(region_sales, values='Revenue', names='Region',
                  title='Revenue Distribution by Region')

    # Summary statistics
    summary = f"""
    ### 📊 Summary Statistics

    - **Total Revenue:** ${total_revenue:,.2f}
    - **Total Units Sold:** {total_units:,}
    - **Average Daily Revenue:** ${avg_revenue:,.2f}
    - **Date Range:** {filtered_df['Date'].min().strftime('%Y-%m-%d')} to {filtered_df['Date'].max().strftime('%Y-%m-%d')}
    """

    # Return sample of data
    data_preview = filtered_df.head(100)

    return summary, fig1, fig2, fig3, data_preview

# Create Gradio interface
with gr.Blocks(title="Sales Analytics Dashboard") as demo:
    gr.Markdown("# 📈 Sales Analytics Dashboard")
    gr.Markdown("Analyze sales data across different products, regions, and time periods.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Filters")
            product_dropdown = gr.Dropdown(
                choices=["All"] + sorted(df['Product'].unique().tolist()),
                value="All",
                label="Product"
            )
            region_dropdown = gr.Dropdown(
                choices=["All"] + sorted(df['Region'].unique().tolist()),
                value="All",
                label="Region"
            )
            start_date = gr.DateTime(
                label="Start Date",
                value=df['Date'].min().to_pydatetime(),
                type="datetime"
            )
            end_date = gr.DateTime(
                label="End Date",
                value=df['Date'].max().to_pydatetime(),
                type="datetime"
            )
            analyze_btn = gr.Button("Analyze", variant="primary")

        with gr.Column(scale=2):
            summary_output = gr.Markdown()

    with gr.Row():
        with gr.Column():
            plot1 = gr.Plot(label="Revenue Trend")
        with gr.Column():
            plot2 = gr.Plot(label="Product Performance")

    with gr.Row():
        plot3 = gr.Plot(label="Regional Distribution")

    gr.Markdown("### 📋 Data Preview (First 100 rows)")
    data_output = gr.Dataframe(
        headers=["Date", "Product", "Region", "Units_Sold", "Revenue"],
        interactive=False,
        wrap=True
    )

    # Connect the analyze button to the function
    analyze_btn.click(
        fn=analyze_sales,
        inputs=[product_dropdown, region_dropdown, start_date, end_date],
        outputs=[summary_output, plot1, plot2, plot3, data_output]
    )

    # Run analysis on load
    demo.load(
        fn=analyze_sales,
        inputs=[product_dropdown, region_dropdown, start_date, end_date],
        outputs=[summary_output, plot1, plot2, plot3, data_output]
    )

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26197b57946d5f6c90.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
